In [1]:
import pymongo as pm 
client = pm.MongoClient( 
'bigdatadb.polito.it', 
ssl=True, 
authSource='carsharing', 
username='ictts', 
password='Ict4SM22!', 
tlsAllowInvalidCertificates=True 
)

In [2]:
db = client['carsharing'] #Choose the DB to use

In [3]:
print(db.list_collection_names())

['enjoy_PermanentBookings', 'hexs_gen', 'enjoy_ActiveBookings', 'ActiveParkings', 'test_collection', 'system.js', 'BigDive7_PermanentBookings', 'enjoy_ActiveParkings', 'PermanentBookings', 'PermanentParkings', 'DebugInfo', 'ictts_enjoy_PermanentBookings', 'ictts_PermanentBookings', 'ActiveBookings', 'enjoy_DebugInfo', 'enjoy_PermanentParkings']


In [4]:
#we should analyze Milano, Miami and minchen as group 18

In [5]:
collections = ['PermanentBookings', 'PermanentParkings']

In [7]:
#How many documents are present in each collection? 
for collection in collections:
    count = db[collection].count_documents({})
    print(collection + ": " + str(count))

PermanentBookings: 28180508
PermanentParkings: 28312676


In [8]:
#Why the number of documents in PermanentParkings and PermanentBooking is similar?

In [9]:
#For which cities the system is collecting data?
cityset = set()

for collection in collections:
    city_c = db[collection].distinct("city")
    for one in city_c:
        cityset.add(one)
    
print(cityset) 

{'New York City', 'Hamburg', 'Seattle', 'Milano', 'Vancouver', 'Frankfurt', 'Rheinland', 'Munchen', 'Torino', 'Wien', 'Roma', 'Firenze', 'Denver', 'Austin', 'Washington DC', 'San Diego', 'Toronto', 'Portland', 'Stuttgart', 'Calgary', 'Berlin', 'Madrid', 'Twin Cities', 'Amsterdam', 'Columbus', 'Montreal'}


In [10]:
#When did each collection start? When did each collection end? 

In [11]:
#What about the time zone of the init_date and init_time timestamps? 
#Which time zone do they refer to?

for collection in collections:
    doc = db[collection].find_one()

    if doc is None:
        print(f"{collection}: collezione vuota")
        continue

    if "init_date" not in doc:
        print(f"{collection}: campo init_date non presente")
        continue

    print(f"{collection}: {doc['init_date']}")

    

PermanentBookings: 2016-12-13 09:38:23
PermanentParkings: 2016-12-13 18:37:38


In [12]:
#What is the total number of cars seen in the whole period in each city? 
cittaList = ["Milano", "Montreal", "Munchen"]

# un set per ogni città
cittaSet = [set(), set(), set()]

for i in range(3):

    for collection in collections:

        # targhe uniche nella singola collection
        plates = db[collection].distinct("plate", {"city": cittaList[i]})

        # aggiungo tutte le targhe trovate al set della città
        cittaSet[i].update(plates)

    # conteggio finale delle targhe uniche della città
    print(f"The number of cars in {cittaList[i]} is {len(cittaSet[i])}")

#prima abbiamo considerato entrambe i database e poi abbiamo solo caonsiderato car2go,
#ricordando che enjoy è solo in ITalia


The number of cars in Milano is 1155
The number of cars in Montreal is 844
The number of cars in Munchen is 586


In [13]:
#How can you estimate the fleet size in a given period, e.g., one week?
import datetime

cittaList = ["Milano", "Montreal", "Munchen"]

# un set per ogni città
cittaSet = [set(), set(), set()]

# intervallo temporale
start = datetime.datetime(2017, 12, 12, 9, 38, 23)
end   = datetime.datetime(2017, 12, 19, 9, 38, 23)

for i in range(3):
    filtro = {
        "$and": [
            {"city": cittaList[i]},
            {
                "$and": [
                    {"init_date": {"$gte": start}},
                    {"init_date": {"$lte": end}}
                ]
            },
            {
                "$and": [
                    {"final_date": {"$gte": start}},
                    {"final_date": {"$lte": end}}
                ]
            }
        ]
    }

    for collection in collections:

        # targhe uniche nella collection per quella città e periodo
        plates = db[collection].distinct("plate", filtro)

        # aggiungo le targhe al set della città
        cittaSet[i].update(plates)

    print(
        f"The number of cars seen in {cittaList[i]} "
        f"is {len(cittaSet[i])} from {start} to {end}"
    )
    

The number of cars seen in Milano is 802 from 2017-12-12 09:38:23 to 2017-12-19 09:38:23
The number of cars seen in Montreal is 430 from 2017-12-12 09:38:23 to 2017-12-19 09:38:23
The number of cars seen in Munchen is 555 from 2017-12-12 09:38:23 to 2017-12-19 09:38:23


In [15]:
#prima troviamo il range di queste date
import datetime

cittaList = ["Milano", "Montreal", "Munchen"]
cittaIniz = []
cittaFin = []

for city in cittaList:

    min_date = datetime.datetime.max
    max_date = datetime.datetime.min

    for collection in collections:

        # data minima della collection per quella città
        first = db[collection].find({"city": city}).sort("init_date", 1).limit(1)

        # data massima della collection per quella città
        last = db[collection].find(
            {"city": city}
        ).sort("init_date", -1).limit(1)

        first = list(first)
        last = list(last)

        if first:
            if first[0]["init_date"] < min_date:
                min_date = first[0]["init_date"]

        if last:
            if last[0]["init_date"] > max_date:
                max_date = last[0]["init_date"]

    cittaIniz.append(min_date)
    cittaFin.append(max_date)
    print(city)
    print("Min:", min_date)
    print("Max:", max_date)
    print()

    

Milano
Min: 2016-12-13 18:37:38
Max: 2018-01-31 14:11:33

Montreal
Min: 2016-12-13 12:37:38
Max: 2018-01-31 08:11:33

Munchen
Min: 2016-12-13 18:37:38
Max: 2018-01-31 14:10:45



In [ ]:
#How does this relate to the total number of vehicles seen in the whole collection? 
from datetime import datetime, timedelta

cities = ["Milano", "Montreal", "Munchen"]

date_ranges = {
    "Milano": (
        datetime(2016, 12, 13, 18, 37, 38),
        datetime(2018, 1, 31, 14, 11, 33)
    ),
    "Montreal": (
        datetime(2016, 12, 13, 12, 37, 38),
        datetime(2018, 1, 31, 8, 11, 33)
    ),
    "Munchen": (
        datetime(2016, 12, 13, 18, 37, 38),
        datetime(2018, 1, 31, 14, 10, 45)
    )
}

for city in cities:

    print(f"\n===== {city} =====")

    start_date, end_date = date_ranges[city]

    current_start = start_date
    week_number = 1

    while current_start < end_date:

        current_end = current_start + timedelta(days=7)

        plates = set()

        for collection in collections:

            cursor = db[collection].find({
                "city": city,
                "init_date": {
                    "$gte": current_start,
                    "$lt": current_end
                }
            })

            for doc in cursor:
                if "plate" in doc:
                    plates.add(doc["plate"])

        print(
            f"Settimana {week_number}: "
            f"{len(plates)} veicoli unici"
        )

        week_number += 1
        current_start = current_end

In [14]:
#How many bookings have been recorded in December 2017 in each city? 
import datetime

cittaList = ["Milano", "Montreal", "Munchen"]

# un set per ogni città
cittaSet = [set(), set(), set()]

# intervallo temporale
start = datetime.datetime(2017, 12, 1, 0, 0, 0)
end   = datetime.datetime(2017, 12, 31, 23, 59, 59)

for i in range(3):
    filtro = {
        "$and": [
            {"city": cittaList[i]},
            {
                "$and": [
                    {"init_date": {"$gte": start}},
                    {"init_date": {"$lte": end}}
                ]
            },
            {
                "$and": [
                    {"final_date": {"$gte": start}},
                    {"final_date": {"$lte": end}}
                ]
            }
        ]
    }

    for collection in collections:

        # targhe uniche nella collection per quella città e periodo
        plates = db[collection].distinct("plate", filtro)

        # aggiungo le targhe al set della città
        cittaSet[i].update(plates)

    print(
        f"The number of cars seen in {cittaList[i]} "
        f"is {len(cittaSet[i])} from {start} to {end}"
    )

The number of cars seen in Milano is 818 from 2017-12-01 00:00:00 to 2017-12-31 23:59:59
The number of cars seen in Montreal is 443 from 2017-12-01 00:00:00 to 2017-12-31 23:59:59
The number of cars seen in Munchen is 558 from 2017-12-01 00:00:00 to 2017-12-31 23:59:59


In [ ]:
#How many bookings have the alternative transportation modes recorded in each city? 

In [41]:
db["PermanentBookings"].find_one()

{'_id': ObjectId('5900e8bc2ad8531539ed0847'),
 'init_fuel': 45,
 'city': 'Vancouver',
 'walking': {'duration': -1, 'distance': -1},
 'vendor': 'car2go',
 'driving': {'duration': -1, 'distance': -1},
 'final_time': 1481650748,
 'plate': '644XNS',
 'engineType': 'CE',
 'init_time': 1481650703,
 'vin': 'WMEEJ3BA8EK762841',
 'smartPhoneRequired': False,
 'interior': 'GOOD',
 'final_fuel': 45,
 'exterior': 'GOOD',
 'init_date': datetime.datetime(2016, 12, 13, 9, 38, 23),
 'final_date': datetime.datetime(2016, 12, 13, 9, 39, 8),
 'init_address': '1694 Duranleau,surface lot next t/Bridges.fits3',
 'final_address': '1694 Duranleau,surface lot next t/Bridges.fits3',
 'origin_destination': {'type': 'LineString',
  'coordinates': [[-123.13606, 49.27269], [-123.13606, 49.27269]]},
 'public_transport': {'duration': -1,
  'distance': -1,
  'arrival_date': -1,
  'arrival_time': -1}}